# Medical insurance charges — Single-model OLS regression

Adapted from **Madrid_Rental_ML_Professor_Aligned.ipynb** to `insurance.csv`.

**Goal:** predict individual medical insurance `charges` from `age`, `sex`, `bmi`, `children`, `smoker`, and `region` using **one conventional ordinary least squares (OLS) model**. The original notebook's clustering, segment-specific models, full-versus-reduced model comparisons, and backward elimination have been removed. The retained approach includes dummy encoding, training-only imputation, VIF, regression tables, prediction metrics, residual diagnostics, and an untouched reserved validation set.

The data contains observed insurance charges; predictions are statistical associations, not causal estimates. Currency units are not specified by the CSV.

## 1. Imports and reproducible settings

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display
import sys

# ROOT = Path.cwd().parent        # notebooks/ -> project/
# sys.path.append(str(ROOT))

# from src.config import load_config

# CONFIG_PATH = ROOT / "config.yaml"

# cfg = load_config(CONFIG_PATH)
# print(cfg)

# DATA_PATH = ROOT / Path(cfg["paths"]["raw_data"])


: 

## 2. Load and audit the data

In [ ]:
df = pd.read_csv(DATA_PATH)
TARGET = 'charges'
NUMERIC = ['age', 'bmi', 'children']
CATEGORICAL = ['sex', 'smoker', 'region']
PREDICTORS = NUMERIC + CATEGORICAL
assert set(PREDICTORS + [TARGET]).issubset(df.columns)
print('Shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print('Missing values:\n', df.isna().sum())
print('Data types:\n', df.dtypes)
display(df.head())


## 3. Conservative cleaning

Convert numerical columns to numeric values and standardize categorical text. Missing target values are removed; predictor missingness, if present, is handled using only the training/development data. No rows are removed for high charges, and no domain-specific outlier corrections are made.

In [ ]:
clean = df.copy()
for col in NUMERIC + [TARGET]:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
for col in CATEGORICAL:
    clean[col] = clean[col].astype('string').str.strip().str.lower()
clean = clean.dropna(subset=[TARGET]).copy()
print('Rows after target cleaning:', len(clean))
print('Missing values after cleaning:\n', clean.isna().sum())
print('Categorical levels:')
for col in CATEGORICAL:
    print(col, sorted(clean[col].dropna().unique().tolist()))


## 4. Exploratory diagnostics

In [ ]:
display(clean[NUMERIC + [TARGET]].describe().T)
plt.figure(figsize=(7, 5))
sns.heatmap(clean[NUMERIC + [TARGET]].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Numeric-variable correlations (descriptive)')
plt.tight_layout()
plt.show()
for col in ['charges', 'age', 'bmi', 'children']:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    sns.histplot(x=clean[col].dropna(), kde=True, ax=axes[0], color="steelblue")
    axes[0].set_title(f'{col} — distribution')
    sns.boxplot(x=clean[col], ax=axes[1])
    axes[1].set_title(f'{col} — boxplot')
    plt.tight_layout()
    plt.show()


In [ ]:
for col in CATEGORICAL:
    summary = clean.groupby(col, dropna=False)[TARGET].agg(['count', 'median', 'mean']).sort_values('median', ascending=False)
    print(f'Charges by {col}:')
    display(summary.round(2))
    plt.figure(figsize=(7, 3.3))
    sns.boxplot(data=clean, x=col, y=TARGET)
    plt.title(f'Charges by {col}')
    plt.tight_layout()
    plt.show()


## 5. Freeze development and reserved samples

Set aside 10% as a final reserved sample. Split the other 90% into training (75% of development) and test (25% of development). The reserved sample is not used to fit the model or choose features.

In [ ]:
development, reserved = train_test_split(clean, test_size=0.10, random_state=SEED)
train, test = train_test_split(development, test_size=0.25, random_state=SEED)
print(f'Train: {len(train)} | Test: {len(test)} | Reserved: {len(reserved)}')
assert len(train) + len(test) + len(reserved) == len(clean)


## 6. Encode predictors and define the single OLS model

Use one-hot encoding with the first level of each categorical variable as its reference category, add an intercept, and impute any missing predictor values with training-only medians. Exactly **one model type and one fixed specification** are used throughout; final refitting on the development set does not introduce another candidate model.

In [ ]:
def raw_features(frame):
    return pd.get_dummies(frame[PREDICTORS], columns=CATEGORICAL, drop_first=False, dtype=float).astype(float)

# Freeze category-reference choices using the training subset only.
train_all = raw_features(train)
reference = {}
for col in CATEGORICAL:
    dummy_cols = sorted(c for c in train_all.columns if c.startswith(f'{col}_'))
    reference[col] = dummy_cols[0] if dummy_cols else None
feature_columns = [c for c in train_all.columns if c not in reference.values()]

# Keep numeric features and available non-reference categories in one fixed design.
def prepare_features(frame, columns=feature_columns):
    return raw_features(frame).reindex(columns=columns, fill_value=0).astype(float)

imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(prepare_features(train)), columns=feature_columns, index=train.index)
X_test = pd.DataFrame(imputer.transform(prepare_features(test)), columns=feature_columns, index=test.index)

# One model: conventional price-level OLS.
def fit_ols(X, y):
    return sm.OLS(y, sm.add_constant(X, has_constant='add')).fit()

def predict_ols(model, X):
    return np.asarray(model.predict(sm.add_constant(X, has_constant='add')))

def scores(y, predicted):
    return {'R2': r2_score(y, predicted),
            'MAE': mean_absolute_error(y, predicted),
            'RMSE': np.sqrt(mean_squared_error(y, predicted))}

model = fit_ols(X_train, train[TARGET])
print('Reference categories:', reference)
print('Predictors:', feature_columns)
print(model.summary())


## 7. VIF and conventional OLS diagnostics

In [ ]:
exog = pd.DataFrame(sm.add_constant(X_train, has_constant='add'), index=X_train.index)
vif = pd.DataFrame({'Term': exog.columns,
                    'VIF': [variance_inflation_factor(exog.to_numpy(dtype=float), i)
                            for i in range(exog.shape[1])]})
display(vif.sort_values('VIF', ascending=False).round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].scatter(model.fittedvalues, model.resid, alpha=.5)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Training residuals vs fitted')
axes[0].set_xlabel('Fitted charges')
axes[0].set_ylabel('Residual charges')
sns.histplot(x=np.asarray(model.resid), kde=True, ax=axes[1], color="steelblue")
axes[1].set_title('Training residuals distribution')
plt.tight_layout()
plt.show()


## 8. Development evaluation (train and test)

In [ ]:
development_results = []
for label, X_part, part in [('Train', X_train, train), ('Test', X_test, test)]:
    pred = predict_ols(model, X_part)
    development_results.append({'Split': label, 'N': len(part), **scores(part[TARGET], pred)})
    plt.figure(figsize=(6, 4))
    plt.scatter(part[TARGET], pred, alpha=.5)
    limits = [min(part[TARGET].min(), pred.min()), max(part[TARGET].max(), pred.max())]
    plt.plot(limits, limits, 'k--')
    plt.title(f'{label}: actual vs predicted charges')
    plt.xlabel('Actual charges')
    plt.ylabel('Predicted charges')
    plt.tight_layout()
    plt.show()
development_scores = pd.DataFrame(development_results)
display(development_scores.round(3))
development_scores.to_csv(OUTPUT_DIR / 'development_scores.csv', index=False)


## 9. Refit the same model on development data and evaluate reserved data once

Following the original notebook's final-validation structure, refit the **same OLS formula** and preprocessing on all development rows, then evaluate only once on the untouched reserved subset.

In [ ]:
final_imputer = SimpleImputer(strategy='median')
X_dev = pd.DataFrame(final_imputer.fit_transform(prepare_features(development)), columns=feature_columns, index=development.index)
X_reserved = pd.DataFrame(final_imputer.transform(prepare_features(reserved)), columns=feature_columns, index=reserved.index)
final_model = fit_ols(X_dev, development[TARGET])
reserved_pred = predict_ols(final_model, X_reserved)
reserved_scores = pd.DataFrame([{'Model': 'OLS', 'N_development': len(development),
                                 'N_reserved': len(reserved), **scores(reserved[TARGET], reserved_pred)}])
display(reserved_scores.round(3))
reserved_scores.to_csv(OUTPUT_DIR / 'reserved_scores.csv', index=False)
reserved_predictions = reserved[PREDICTORS + [TARGET]].copy()
reserved_predictions['predicted_charges'] = reserved_pred
reserved_predictions['residual'] = reserved_predictions[TARGET] - reserved_predictions['predicted_charges']
reserved_predictions['absolute_error'] = reserved_predictions['residual'].abs()
reserved_predictions.to_csv(OUTPUT_DIR / 'reserved_predictions.csv', index=False)
plt.figure(figsize=(6, 4))
plt.scatter(reserved[TARGET], reserved_pred, alpha=.6)
limits = [min(reserved[TARGET].min(), reserved_pred.min()), max(reserved[TARGET].max(), reserved_pred.max())]
plt.plot(limits, limits, 'k--')
plt.xlabel('Actual charges')
plt.ylabel('Predicted charges')
plt.title('Reserved sample: actual vs predicted')
plt.tight_layout()
plt.show()


## 10. Final conventional OLS coefficient table

Categorical coefficients compare with the omitted reference level for that variable, controlling for the other predictors. These are associations, not causal effects.

In [ ]:
print('Final OLS summary (fit on all development rows):')
print(final_model.summary())
coefficients = pd.DataFrame({
    'Term': final_model.params.index,
    'Coefficient': final_model.params.values,
    'Std_Error': final_model.bse.values,
    't_statistic': final_model.tvalues.values,
    'p_value': final_model.pvalues.values,
    'CI_low': final_model.conf_int().iloc[:, 0].values,
    'CI_high': final_model.conf_int().iloc[:, 1].values
})
display(coefficients.round(4))
coefficients.to_csv(OUTPUT_DIR / 'OLS_coefficients.csv', index=False)
print('Largest reserved-sample absolute errors:')
display(reserved_predictions.nlargest(5, 'absolute_error'))


## 11. Reproducibility checks and exports

In [ ]:
assert len(clean) == 1338, 'The supplied insurance.csv was expected to contain 1,338 rows.'
assert len(reserved_predictions) == len(reserved)
assert list(X_train.columns) == list(X_dev.columns)
assert len(development_scores) == 2 and len(reserved_scores) == 1
print('Input:', DATA_PATH)
print('One model: OLS, fixed specification')
print('Exports:')
for file in sorted(OUTPUT_DIR.glob('*.csv')):
    print(' -', file)
